In [3]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatibl

In [4]:
#TASK 2
from ortools.sat.python import cp_model

def solve_delivery_schedule():
    model = cp_model.CpModel()

    D1 = model.NewIntVar(1, 3, "D1")
    D2 = model.NewIntVar(1, 3, "D2")
    D3 = model.NewIntVar(1, 3, "D3")
    D4 = model.NewIntVar(1, 3, "D4")

    model.Add(D1 != D2)
    model.Add(D3 < D4)
    model.Add(D2 != 1)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print("D1 =", solver.Value(D1))
        print("D2 =", solver.Value(D2))
        print("D3 =", solver.Value(D3))
        print("D4 =", solver.Value(D4))
    else:
        print("No solution found.")

solve_delivery_schedule()

D1 = 1
D2 = 2
D3 = 1
D4 = 2


In [1]:
#TASK 4
from ortools.sat.python import cp_model

def solve_vehicle_assignment():
    model = cp_model.CpModel()

    #vehicles mapping: V1=1, V2=2, V3=3

    #vars
    L1 = model.NewIntVar(1, 3, "L1")
    L2 = model.NewIntVar(1, 3, "L2")
    L3 = model.NewIntVar(1, 3, "L3")
    L4 = model.NewIntVar(1, 3, "L4")
    L5 = model.NewIntVar(1, 3, "L5")

    locations = [L1, L2, L3, L4, L5]

    #constraint:L1 and L2 not same vehicle
    model.Add(L1 != L2)

    #constraint: L3 must be V3
    model.Add(L3 == 3)
    v1_count = sum([model.NewBoolVar(f"v1_{i}") for i in range(5)])
    v2_count = sum([model.NewBoolVar(f"v2_{i}") for i in range(5)])
    v3_count = sum([model.NewBoolVar(f"v3_{i}") for i in range(5)])

    v1_bools = []
    v2_bools = []
    v3_bools = []

    for i, loc in enumerate(locations):
        b1 = model.NewBoolVar(f"L{i+1}_is_V1")
        b2 = model.NewBoolVar(f"L{i+1}_is_V2")
        b3 = model.NewBoolVar(f"L{i+1}_is_V3")

        model.Add(loc == 1).OnlyEnforceIf(b1)
        model.Add(loc != 1).OnlyEnforceIf(b1.Not())

        model.Add(loc == 2).OnlyEnforceIf(b2)
        model.Add(loc != 2).OnlyEnforceIf(b2.Not())

        model.Add(loc == 3).OnlyEnforceIf(b3)
        model.Add(loc != 3).OnlyEnforceIf(b3.Not())

        v1_bools.append(b1)
        v2_bools.append(b2)
        v3_bools.append(b3)

    model.Add(sum(v1_bools) <= 2)
    model.Add(sum(v2_bools) <= 2)
    model.Add(sum(v3_bools) <= 3)

    diff = model.NewIntVar(-5, 5, "diff")
    model.Add(diff == sum(v1_bools) - sum(v2_bools))
    model.AddAbsEquality(model.NewIntVar(0, 5, "abs_diff"), diff)
    model.AddAbsEquality(model.NewIntVar(0, 5, "abs_diff2"), diff)
    model.AddAbsEquality(diff, diff)
    model.Add(diff <= 1)
    model.Add(diff >= -1)

    b = model.NewBoolVar("L4_is_V1")
    model.Add(L4 == 1).OnlyEnforceIf(b)
    model.Add(L4 != 1).OnlyEnforceIf(b.Not())
    model.Add(L5 == 1).OnlyEnforceIf(b)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print("L1 =", solver.Value(L1))
        print("L2 =", solver.Value(L2))
        print("L3 =", solver.Value(L3))
        print("L4 =", solver.Value(L4))
        print("L5 =", solver.Value(L5))
    else:
        print("No solution found.")

solve_vehicle_assignment()

L1 = 2
L2 = 3
L3 = 3
L4 = 3
L5 = 1


In [2]:
#TASK 5
from ortools.sat.python import cp_model

def solve_exam_schedule():
    model = cp_model.CpModel()

    #vars
    Math = model.NewIntVar(1, 6, "Math")
    Physics = model.NewIntVar(1, 6, "Physics")
    AI = model.NewIntVar(1, 6, "AI")
    OS = model.NewIntVar(1, 6, "OS")
    DB = model.NewIntVar(1, 6, "DB")

    exams = [Math, Physics, AI, OS, DB]

    model.Add(Math != Physics)
    model.Add(AI != OS)
    model.Add(OS != DB)

    diff = model.NewIntVar(-6, 6, "diff")
    model.Add(diff == AI - OS)
    abs_diff = model.NewIntVar(0, 6, "abs_diff")
    model.AddAbsEquality(abs_diff, diff)
    model.Add(abs_diff >= 2)

    model.Add(Math < DB)

    #slot capacity (max 2 exams per slot)
    for slot in range(1, 7):
        bools = []
        for i, exam in enumerate(exams):
            b = model.NewBoolVar(f"exam_{i}_slot_{slot}")
            model.Add(exam == slot).OnlyEnforceIf(b)
            model.Add(exam != slot).OnlyEnforceIf(b.Not())
            bools.append(b)
        model.Add(sum(bools) <= 2)

    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print("Math =", solver.Value(Math))
        print("Physics =", solver.Value(Physics))
        print("AI =", solver.Value(AI))
        print("OS =", solver.Value(OS))
        print("DB =", solver.Value(DB))
    else:
        print("No solution found.")

solve_exam_schedule()

Math = 2
Physics = 1
AI = 4
OS = 6
DB = 5
